**Why does this matter for quant financ**

Computers don't do maths the way you learnt it in school. A trading system that silently miscalculates a variance, overflows a position size, or loses precision in a compounding calculation isn't just academically wrong — it can lose real money. The 2010 Flash Crash, the Knight Capital disaster, and countless smaller quant blowups all have numerical errors somewhere in their ancestry. This module is about knowing where the landmines are. 💣

**Part 1 — Floating Point: Why Computers Can't Count**

Computers store numbers in binary floating point — and most decimal numbers can't be represented exactly in binary. This causes silent rounding errors.

In [1]:
import numpy as np

# This should be 0 — is it?
print(0.1 + 0.2 - 0.3)
# 5.551115123125783e-17  ← NOT zero! 😱

# Same issue in NumPy
a = np.float64(0.1) + np.float64(0.2)
print(a == 0.3)  # False ❌

5.551115123125783e-17
False


Why Does This Happen? 🤔

Just like 1/3 = 0.3333... can't be written exactly in decimal, 0.1 can't be written exactly in binary. The computer stores the closest approximation — and those tiny errors accumulate.

In [2]:
# The right way to compare floats
print(np.isclose(0.1 + 0.2, 0.3))  # True ✅
print(np.allclose(arr1, arr2))      # for arrays

True


NameError: name 'arr1' is not defined

🏦 Quant context: Never use == to compare prices or returns. 

Always use np.isclose(). A risk system that checks if pnl == 0 could silently miss non-zero P&L due to floating point error.

📐 Part 2 — dtypes & Precision

NumPy arrays have a dtype that controls precision and memory:

In [4]:
# float64 — the default, 64-bit double precision
a = np.array([1.0, 2.0, 3.0])
print(a.dtype)  # float64

# float32 — half the memory, less precision
b = np.array([1.0, 2.0, 3.0], dtype=np.float32)

# int32 vs int64 — affects overflow behaviour
c = np.array([1, 2, 3], dtype=np.int32)

float64


**The precision trade off**
```
dtype       Memory      Max Value       Use Case
float64     8 bytes     ~1.8 × 10³⁰⁸    ✅ Default for quant work
float32     4 bytes     ~3.4 × 10³⁸     ML model weights, large datasets
int32       4 bytes     ~2.1 × 10⁹      Small integers
int64       8 bytes     ~9.2 × 10¹⁸     Large integers, timestamps
💡 Rule of thumb: Always use float64 for financial calculations unless you have a specific reason not to. The extra precision is almost always worth it.

⚠️ Part 3 — Overflow & Underflow

Overflow — a number gets too large for its dtype to represent:

In [5]:
# int32 overflow — silent and dangerous!
a = np.array([2_000_000_000], dtype=np.int32)
print(a * 2)  # -294967296 ← WRONG, wrapped around! 😱

# float64 overflow
print(np.float64(1e308) * 10)  # inf — at least it's obvious!

[-294967296]
inf


/var/folders/3t/_m90qmv50fxb_842_2vl8_qw0000gn/T/ipykernel_12240/3885302324.py:6: RuntimeWarning: overflow encountered in scalar multiply
  print(np.float64(1e308) * 10)  # inf — at least it's obvious!


Underflow — a number gets too small and collapses to zero:

In [6]:
# Tiny probability that disappears
p = np.float64(1e-300)
print(p * p)  # 0.0 ← should be 1e-600, but underflows to zero

0.0


🏦 Quant context: Underflow is particularly nasty in probability calculations — multiplying many small probabilities together (e.g. in a likelihood function) can silently collapse to zero. The fix is to work in log space:

In [ ]:
# ❌ Underflows to zero for long return series
probs = np.array([0.51, 0.49, 0.52, 0.48, ...])
likelihood = np.prod(probs)  # → 0.0

# ✅ Sum log probabilities instead
log_likelihood = np.sum(np.log(probs))  # stays numerically stable

🎯 Part 4 — Catastrophic Cancellation

This is the sneakiest one. When you subtract two nearly-equal large numbers, you lose significant digits:

In [9]:
a = np.float64(1_000_000.001)
b = np.float64(1_000_000.000)
print(a - b)  # 9.313225746154785e-07 ← imprecise!

0.0010000000474974513


The Classic Example — Naive Variance
The textbook formula for variance that loses precision:

In [10]:
# ❌ Naive (unstable) variance calculation
def naive_variance(x):
    n = len(x)
    return (np.sum(x**2) - np.sum(x)**2 / n) / n

# ✅ Stable — subtract mean first (Welford's method concept)
def stable_variance(x):
    return np.var(x)  # NumPy uses a stable algorithm internally!

# See the difference on large numbers
data = np.array([1e8 + 1, 1e8 + 2, 1e8 + 3, 1e8 + 4])
print("Naive:", naive_variance(data))    # potentially imprecise
print("Stable:", stable_variance(data))  # 1.25 ← correct

Naive: 2.0
Stable: 1.25


💡 This is precisely why np.var() exists — NumPy's implementation uses numerically stable algorithms so you don't have to implement them yourself!

**Part 5 - Numerical Stability in Practise**

In [ ]:
# ✅ Use np.isclose() not == for float comparison
np.isclose(a, b, rtol=1e-9)

# ✅ Use np.log() for products of many small numbers
np.sum(np.log(probabilities))  # instead of np.prod(probabilities)

# ✅ Use np.round() only at display stage, NEVER mid-calculation
result = some_calculation()
print(np.round(result, 4))  # ✅ display only

# ✅ Check for inf and nan after calculations
np.isinf(result)   # True if any element is infinity
np.isnan(result)   # True if any element is NaN (Not a Number)
np.any(np.isnan(result))  # useful check after large calculations

NameError: name 'probabilities' is not defined

Where Do NaN and inf Come From?

In [ ]:
print(np.log(0))      # -inf  ← log of zero
print(np.log(-1))     # nan   ← log of negative number
print(0 / 0)          # nan   ← zero divided by zero (Python raises error)
print(np.float64(0) / np.float64(0))  # nan in NumPy
print(1 / np.float64(0))  # inf in NumPy

🏦 Quant context: NaN in a returns series will silently poison any calculation it touches — np.mean([0.01, np.nan, 0.02]) returns nan. Always validate data with np.isnan() checks before running analytics!

**Tasks**

Task 1: Demonstrate floating point imprecision — show that
          0.1 + 0.2 != 0.3, then fix it with np.isclose()

 Task 2: Show int32 overflow by multiplying a large number
          then repeat with int64 and compare results

 Task 3: Write both the naive and stable variance functions from above
          and test them on: np.array([1e8+1, 1e8+2, 1e8+3, 1e8+4])
          The correct variance is 1.25

 Task 4: Take this returns series and:
          
          a) Find any NaN or inf values
          
          b) Replace NaN with the mean return (hint: np.nanmean())
returns = np.array([0.01, -0.02, np.nan, 0.03, np.inf, -0.01, 0.02])

 Task 5: A stock compounds daily. Show why multiplying 252 daily
          probabilities together underflows, then fix it using log space

daily_probs = np.full(252, 0.501)  # slight edge every day

In [43]:
#Task 1 - Demonstrating floating point imprecision
print("Task 1")
imprecise_sum = 0.1 + 0.2
print("Using == : ", imprecise_sum == 0.3) #This should output as false !
precise_sum = np.isclose(imprecise_sum, 0.3)
print("Using np.isclose() : ", precise_sum)

#Task 2 - Showing int32 overflow with multiplying large numbers
    #using int32
print("\nTask 2")
a = np.array([2_000_000_000], dtype=np.int32)
print("Using int32: ", a * 2) 

# # int64 overflow
b = np.array([2_000_000_000], dtype=np.int64)
print("Using int64: ", b * 2)

#Task 3 - Using both the naive and stable variance functions
print("\nTask 3")
def naive_variance(x):
    n = len(x)
    return (np.sum(x**2) - np.sum(x)**2 / n) / n
def stable_variance(x):
    return np.var(x)

data = np.array([1e8+1, 1e8+2, 1e8+3, 1e8+4])
print("Naive:", naive_variance(data)) 
print("Stable:", stable_variance(data)) 
    
#Task 4a
print("\nTask 4a")
returns = np.array([0.01, -0.02, np.nan, 0.03, np.inf, -0.01, 0.02])
print(np.isnan(returns))
print(np.isinf(returns))

#Task 4b
print("\nTask 4b")
print(np.nanmean(returns)) # This isn't enough to replace the nan! np.nanmean is just the mean ignoreing the nan
mean_val = np.nanmean(returns[~np.isinf(returns)])  # exclude inf too!
returns_clean = np.where(np.isnan(returns), mean_val, returns)
print(returns_clean)

#Task 5
print("\nTask 5")
daily_probs = np.full(252, 0.501) 
print(daily_probs)

product = 1
for prob in daily_probs:
    product *= prob

print(product)
prod_daily_probs = np.prod(daily_probs)

print(product == prod_daily_probs)
print(np.isclose(product, prod_daily_probs))
np.sum(np.log(daily_probs))


Task 1
Using == :  False
Using np.isclose() :  True

Task 2
Using int32:  [-294967296]
Using int64:  [4000000000]

Task 3
Naive: 2.0
Stable: 1.25

Task 4a
[False False  True False False False False]
[False False False False  True False False]

Task 4b
inf
[ 0.01  -0.02   0.006  0.03     inf -0.01   0.02 ]

Task 5
[0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501
 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501
 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501
 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501
 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501
 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501
 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501
 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501
 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501 0.501
 0.501 0.501 0.501 0.501 0.5

np.float64(-174.16959283011263)

🧪 Self-Check Questions

Q: Why should you never use == to compare two floats in a financial system?<br>
A: floats can't represent most decimal numbers exactly in binary, so tiny rounding errors accumulate. 0.1 + 0.2 literally stores a different bit pattern than 0.3.


Q:What's the difference between overflow and underflow? Give a quant example of each.<br>
A: 
    Overflow = number exceeds the maximum a dtype can store → wraps around (int) or becomes inf (float)
    Underflow = number is too small → collapses silently to 0.0
    These are opposites, not just "over/under calculating"!

Q: What is catastrophic cancellation and why is the naive variance formula vulnerable to it?<br>
A: Catastrophic cancellation is when you subtract two nearly equal large numbers — the large parts cancel out, leaving only the imprecise bits. The naive variance formula is vulnerable because it computes sum(x²) - sum(x)² — two huge numbers whose difference should be tiny.

Q: Why does np.var() exist if you can calculate variance manually?<br>
A: p.var() exists because it uses a numerically stable algorithm internally — it subtracts the mean first before squaring, avoiding catastrophic cancellation entirely.

Q: A NaN appears in your returns array mid-pipeline. What caused it and how do you detect and handle it?<br>
A: NaN comes from operations like log(-1), 0/0, or np.inf - np.inf — not from numbers close to zero. Detection is np.isnan(), and the fix is np.nanmean() to impute.



That's the spirit! 😄 Let's go!

---

## 🧪 Module 8 Quiz — Numerical Precision & Stability

---

**Q1.** What does this print, and why?

```python
a = np.array([2_100_000_000], dtype=np.int32)
print(a * 2)
```

---

**Q2.** What's wrong with this comparison, and how do you fix it?

```python
returns = np.float64(0.1) + np.float64(0.2)
if returns == 0.3:
    print("Equal")
```

---

**Q3.** Why does this collapse to zero, and what's the fix?

```python
daily_probs = np.full(252, 0.499)
likelihood = np.prod(daily_probs)
print(likelihood)  # 0.0
```

---

**Q4.** What will `np.mean()` return on this array, and how do you fix it?

```python
returns = np.array([0.01, -0.02, np.nan, 0.03, -0.01])
print(np.mean(returns))
```

---

**Q5.** 🏦 **Quant Finance scenario** — A junior analyst writes this variance calculation on a series of large prices. What's the risk, and what's the fix?

```python
prices = np.array([1e8+1, 1e8+5, 1e8+3, 1e8+2])
variance = (np.sum(prices**2) - np.sum(prices)**2 / len(prices)) / len(prices)
```

---

Take your time! 🧠

**My answers**

Q1: It will print a number slightly over 4_200_000_000, this is because of overflow where the number exceedes the maximum a dtype can store. In this case, you can use int64

Q2: This will not print equal - this is because numbers, specifically floats, (despite how they are visualised) are stored as slight approximations of numbers. The right way to compare floats is to use np.isclose(), or if it's an array, np.allclose()

Q3: It will collapose to 0 because the number is too small. The way to fix this is to log them, and them sum them

Q4: np.mean() will return the average of all the numbers apart from the np.nan. What should be done instead is to log the array and then sum them.

Q5: The risk is that it does a naive calculation and underflow problems will occur (where the number is so small it collapses to 0. This means that the resulting variance will, in effect, be really close to 0. What should be done instead is to use np.var(prices)

**Marking**

Good effort Clark — let me mark these up carefully! 🔍

**Q1 — ⚠️ Partially right.**
You correctly identified overflow and the fix, but the output isn't *slightly over* 4.2 billion — int32 **wraps around to a negative number**! The max int32 is ~2.1 billion, so exceeding it flips the sign entirely. Run it and see! 😮

**Q2 — ✅ Perfect.** Textbook answer.

**Q3 — ✅ Perfect.** Underflow → log space → sum. Exactly right.

**Q4 — ❌ Careful here!**
`np.mean()` on an array containing `NaN` returns **`NaN`** — it poisons the entire result. And the fix is **`np.nanmean()`**, not logging! You've mixed up the log space solution from Q3 with this one. Two different problems, two different fixes! 🐍

**Q5 — ⚠️ Right fix, wrong problem name!**
`np.var()` is the correct fix, but the risk here is **catastrophic cancellation** — not underflow. Subtracting two enormous nearly-equal numbers (`sum(x²) - sum(x)²`) destroys precision. Underflow is about numbers collapsing to zero, which isn't what's happening here.

---

## 📊 Final Score: 3/5

Q4 and Q5 show the self-check gaps are still lingering. I'd strongly suggest revisiting Parts 3, 4 and 5 before the capstone Clark — you want these concepts sharp! 💪 Agreed?

**Notes on the difference between using np.nan() and np.sum(np.log())**

Great question — these two are actually solving completely different problems! They just happen to both deal with awkward data. 🎯

---

## `np.nanmean()` — Handling Missing Data

**Problem it solves:** `NaN` values in your array that would poison any calculation.

**What it does:** Computes the mean while simply **ignoring** NaN values.

```python
returns = np.array([0.01, -0.02, np.nan, 0.03, -0.01])

np.mean(returns)     # nan ❌ — NaN poisons everything
np.nanmean(returns)  # 0.0025 ✅ — ignores the NaN
```

**When to use it:** Data cleaning — when your dataset has missing values you want to skip over.

---

## `np.sum(np.log())` — Handling Numerical Underflow

**Problem it solves:** Multiplying many small probabilities together collapses to zero.

**What it does:** Converts a product into a sum using the log identity:
```
log(a × b × c) = log(a) + log(b) + log(c)
```

```python
daily_probs = np.full(252, 0.501)

np.prod(daily_probs)              # 0.0 ❌ — underflows to zero
np.sum(np.log(daily_probs))       # -1.96 ✅ — stable!
```

**When to use it:** Probability calculations — when multiplying many small numbers together.

---

## The Key Distinction

| | `np.nanmean()` | `np.sum(np.log())` |
|---|---|---|
| Solves | Missing data | Numerical underflow |
| Used in | Data cleaning | Probability/likelihood |
| Changes result? | Slightly (ignores NaNs) | No — mathematically equivalent |

Think of it this way — `np.nanmean()` is a **data quality** tool, `np.sum(np.log())` is a **numerical stability** tool. 💡